# vLLM AsyncLLMEngine + Qwen — Colab GPU (in-process, streaming control)

Same Qwen2.5 backend as `vllm_qwen_colab.ipynb`, but built on **`AsyncLLMEngine`** in-process
instead of `vllm serve`. This gives control the OpenAI server does not expose:

- **Token streaming** you can act on mid-generation (SSE endpoint + a direct demo).
- **Early-abort** a generation (`engine.abort(request_id)`) — cancel a weak branch to save tokens.
- **Per-token logprobs** — usable as a confidence signal.
- **Guided JSON** (xgrammar) + **prefix caching** (KV reuse across siblings), same as the server.

It still exposes an OpenAI-compatible `/v1/chat/completions` (+ `/v1/models`, `/health`) so it is a
drop-in for the existing model-service / planner wiring — plus `stream:true` SSE.

**Run:** GPU runtime > Run all > copy `TUNNEL_URL` + `API_KEY` from the banner.

> Note: vLLM's API shifts between versions. Cells flag the version-sensitive imports
> (`AsyncLLMEngine`, `GuidedDecodingParams`). If an import fails, `pip show vllm` and adjust.


In [ ]:
# -- 1 - GPU check & model auto-selection --------------------------------------
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
assert gpu, "No GPU - Runtime > Change runtime type > select a GPU (L4 recommended)"
print("GPU:", gpu)

gpu_name = gpu.lower()
if "a100" in gpu_name:
    MODEL, DTYPE, MAX_LEN = "Qwen/Qwen2.5-14B-Instruct-AWQ", "auto", 16384
elif "l4" in gpu_name:
    MODEL, DTYPE, MAX_LEN = "Qwen/Qwen2.5-7B-Instruct", "auto", 8192
elif "t4" in gpu_name:
    MODEL, DTYPE, MAX_LEN = "Qwen/Qwen2.5-7B-Instruct-AWQ", "half", 8192
else:
    MODEL, DTYPE, MAX_LEN = "Qwen/Qwen2.5-7B-Instruct-AWQ", "auto", 8192
print(f"Selected: {MODEL}  dtype={DTYPE}  max_model_len={MAX_LEN}")


In [ ]:
# -- 2 - Install vLLM (CUDA-matched) + serving deps + cloudflared (~3-6 min) -----
# Same CUDA-matched vLLM install as the server notebook (Colab's driver is CUDA 12.x,
# the newest PyPI wheel links CUDA 13). Then fastapi/uvicorn/nest_asyncio to host the
# AsyncLLMEngine inside this notebook's event loop.
import json, re, subprocess, urllib.request

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
drv = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", smi)
assert drv, "could not read driver CUDA version from nvidia-smi"
drv_major = int(drv.group(1))
print(f"driver supports CUDA {drv.group(1)}.{drv.group(2)}")

%pip install -q uv
if drv_major >= 13:
    !uv pip install --system -q -U vllm --torch-backend=auto
else:
    rel = json.load(urllib.request.urlopen(
        "https://api.github.com/repos/vllm-project/vllm/releases/latest"))
    cu_wheels = {}
    for asset in rel["assets"]:
        m = re.search(r"\+cu(\d{3})-cp\d+-abi3-.*x86_64\.whl$", asset["browser_download_url"])
        if m:
            cu_wheels[int(m.group(1))] = asset["browser_download_url"]
    usable = sorted(v for v in cu_wheels if v // 10 == drv_major)
    assert usable, f"release {rel['tag_name']} has no cu{drv_major}x wheel (found: {sorted(cu_wheels)})"
    cu = usable[-1]; wheel = cu_wheels[cu]
    print(f"installing {wheel.rsplit('/', 1)[-1]}  (torch index: cu{cu})")
    !uv pip install --system -q "{wheel}" --extra-index-url https://download.pytorch.org/whl/cu{cu}

!uv pip install --system -q fastapi "uvicorn[standard]" nest_asyncio
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("install done")


In [ ]:
# -- 3 - Config -----------------------------------------------------------------
import secrets
PORT = 8000
API_KEY = "kb-" + secrets.token_urlsafe(16)   # tunnel is public - always require a key
print("API key:", API_KEY)


In [ ]:
# -- 4 - Build AsyncLLMEngine (first run downloads weights - 5-15 min) -----------
import asyncio, os, uuid, nest_asyncio
nest_asyncio.apply()                      # one event loop shared by engine + uvicorn + demos

from vllm import AsyncLLMEngine, SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
try:
    from vllm.sampling_params import GuidedDecodingParams   # guided JSON (xgrammar)
except Exception:
    GuidedDecodingParams = None
    print("WARN: GuidedDecodingParams unavailable in this vLLM - json_schema will be best-effort")

if "t4" in gpu_name:
    os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"

engine = AsyncLLMEngine.from_engine_args(AsyncEngineArgs(
    model=MODEL, dtype=DTYPE, max_model_len=MAX_LEN,
    gpu_memory_utilization=0.92,
    enable_prefix_caching=True,           # KV reuse across siblings / repeated prefixes
))

_tokenizer = None
async def get_tokenizer():
    global _tokenizer
    if _tokenizer is None:
        _tokenizer = await engine.get_tokenizer()
    return _tokenizer

async def _prompt_from(messages):
    tok = await get_tokenizer()
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def _sampling(n=1, temperature=0.7, max_tokens=512, json_schema=None, logprobs=None):
    gd = GuidedDecodingParams(json=json_schema) if (json_schema and GuidedDecodingParams) else None
    return SamplingParams(n=n, temperature=temperature, max_tokens=max_tokens,
                          logprobs=logprobs, guided_decoding=gd)

print("engine ready:", MODEL)


In [ ]:
# -- 5 - DEMO: token streaming + early-abort + logprobs --------------------------
# This is what AsyncLLMEngine gives you over the OpenAI server: act on tokens as they
# arrive, abort mid-generation, and read per-token logprobs.
import math

async def stream_with_abort(user_text, max_tokens=200, abort_after=40):
    prompt = await _prompt_from([{"role": "user", "content": user_text}])
    sp = _sampling(temperature=0.7, max_tokens=max_tokens, logprobs=1)
    rid = str(uuid.uuid4())
    seen, last_lp = 0, None
    async for out in engine.generate(prompt, sp, rid):
        o = out.outputs[0]
        print(o.text[seen:], end="", flush=True); seen = len(o.text)
        if o.logprobs:                              # mean logprob so far = a confidence signal
            lps = [next(iter(d.values())).logprob for d in o.logprobs if d]
            last_lp = sum(lps) / len(lps) if lps else None
        if len(o.token_ids) >= abort_after:
            await engine.abort(rid)                 # <-- prune this generation early
            print(f"\n[aborted after {len(o.token_ids)} tokens | mean_logprob={last_lp:.3f} "
                  f"-> conf~{math.exp(last_lp):.3f}]")
            return
    print(f"\n[finished | mean_logprob={last_lp}]")

await stream_with_abort("Explain how vLLM continuous batching works, in detail.")


In [ ]:
# -- 6 - OpenAI-compatible server on AsyncLLMEngine (+ SSE stream) ---------------
# Drop-in for model-service / planner: /v1/chat/completions, /v1/models, /health.
# Supports n, temperature, max_tokens, response_format json_schema, and stream:true.
import json, threading, time, subprocess, urllib.request
import uvicorn
from fastapi import FastAPI, Request, Header, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse

app = FastAPI(title="vllm-async-engine")

def _auth(authorization):
    if authorization != f"Bearer {API_KEY}":
        raise HTTPException(401, "bad api key")

@app.get("/health")
def health(): return {"status": "ok"}

@app.get("/v1/models")
def models(authorization: str = Header(None)):
    _auth(authorization)
    return {"object": "list", "data": [{"id": MODEL, "object": "model", "owned_by": "vllm"}]}

@app.post("/v1/chat/completions")
async def chat_completions(req: Request, authorization: str = Header(None)):
    _auth(authorization)
    body = await req.json()
    messages = body["messages"]
    n = int(body.get("n", 1)); temp = float(body.get("temperature", 0.7))
    mt = int(body.get("max_tokens", 512)); stream = bool(body.get("stream", False))
    rf = body.get("response_format") or {}
    schema = rf.get("json_schema", {}).get("schema") if rf.get("type") == "json_schema" else None
    prompt = await _prompt_from(messages)
    rid = str(uuid.uuid4())

    if not stream:
        sp = _sampling(n=n, temperature=temp, max_tokens=mt, json_schema=schema)
        final = None
        async for out in engine.generate(prompt, sp, rid):
            final = out
        return JSONResponse({
            "id": rid, "object": "chat.completion", "model": MODEL,
            "choices": [{"index": i, "message": {"role": "assistant", "content": o.text},
                         "finish_reason": "stop"} for i, o in enumerate(final.outputs)],
        })

    sp = _sampling(n=1, temperature=temp, max_tokens=mt, json_schema=schema)
    async def sse():
        seen = 0
        async for out in engine.generate(prompt, sp, rid):
            delta = out.outputs[0].text[seen:]; seen += len(delta)
            if delta:
                chunk = {"id": rid, "object": "chat.completion.chunk", "model": MODEL,
                         "choices": [{"index": 0, "delta": {"content": delta}}]}
                yield f"data: {json.dumps(chunk)}\n\n"
        yield "data: [DONE]\n\n"
    return StreamingResponse(sse(), media_type="text/event-stream")

# run uvicorn on THIS event loop (shares it with the engine - avoids cross-loop futures)
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning", loop="asyncio")
server = uvicorn.Server(config)
asyncio.get_event_loop().create_task(server.serve())
time.sleep(3)
print(urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=5).read())


In [ ]:
# -- 7 - Cloudflare tunnel + banner ----------------------------------------------
import re, subprocess, time

cf = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}",
                       "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = cf.stdout.readline()
    if not line: continue
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m: url = m.group(0); break
assert url, "tunnel URL not found - check cloudflared output"
print("=" * 70)
print(f"  TUNNEL_URL : {url}")
print(f"  API_KEY    : {API_KEY}")
print("=" * 70)
print("Point model-service at it:  REMOTE_LLM_URL=<TUNNEL_URL> REMOTE_LLM_KEY=<API_KEY>")


In [ ]:
# -- 8 - Keep-alive (keep this tab open) -----------------------------------------
import time
while True:
    time.sleep(60)
